In [ ]:
%env CUDA_VISIBLE_DEVICES=1
%load_ext autoreload
%autoreload 2
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from openTSNE import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

from ne_aligned_utils.utils import get_bipartite_graph, sparsify_affinities


In [ ]:
dataset = "nCoV"
res_path = os.path.join("data", dataset)
fig_path = os.path.join("results", "figures")

In [ ]:

# -------------------- I/O --------------------
# After R conversion, point this to the converted AnnData file:
adata = sc.read_h5ad(os.path.join(res_path, "Final_nCoV_0716_upload.h5ad"))

# If raw UMI counts are present, keep them in a dedicated layer for HVG
# (zellkonverter usually stores counts in .X; sometimes also in .layers["counts"]).
if "counts" in adata.layers:
    pass  # already present
else:
    adata.layers["counts"] = adata.X.copy()

In [ ]:
# -------------------- Build/align metadata (batch & cell_type) --------------------
# Your R used: data0@meta.data$batch and $cell_type
# After conversion, these should be in adata.obs with similar names.
obs = adata.obs
# Try common column names; adjust if your h5ad differs.
batch_col_candidates = ["batch", "Batch", "batchname"]
ctype_col_candidates = ["cell_type", "celltype", "final_cell_label"]

def pick_col(cands):
    for c in cands:
        if c in obs.columns:
            return c
    raise KeyError(f"None of {cands} found in adata.obs. Available: {list(obs.columns)}")

batch_col = pick_col(batch_col_candidates)
ctype_col = pick_col(ctype_col_candidates)

adata.obs["method"] = obs[batch_col].astype(str)
adata.obs["cell_type"] = obs[ctype_col].astype("category")

In [ ]:
# -------------------- Seurat-like CreateSeuratObject filters --------------------
# min.features=10 (cells with at least 10 detected genes)
sc.pp.filter_cells(adata, min_genes=10)
# min.cells=3 (genes detected in at least 3 cells)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
# -------------------- Select the two methods and relabel --------------------
# In your R: X1 <- "COV-5-D13", X2 <- "COV-5-D1"
keep_methods = ["COV-5-D13", "COV-5-D1"]
adata = adata[adata.obs["method"].isin(keep_methods)].copy()
adata.obs["method"] = adata.obs["method"].map({
    "COV-5-D13": "data1",
    "COV-5-D1":  "data2"
}).astype("category")

In [ ]:
# -------------------- Gene intersection between the two subsets --------------------
adata1 = adata[adata.obs["method"] == "data1"].copy()
adata2 = adata[adata.obs["method"] == "data2"].copy()

genes1 = pd.Index(adata1.var_names)
genes2 = pd.Index(adata2.var_names)
shared_genes = genes1.intersection(genes2)

# enforce identical order in both
adata1 = adata1[:, shared_genes].copy()
adata2 = adata2[:, shared_genes].copy()

# Recombine (like cbind on matched genes)
adata_comb = sc.concat({"data1": adata1, "data2": adata2}, axis=0, join="outer", label="method_concat")
# The concat gives a helper column; restore original method from obs:
adata_comb.obs["method"] = adata_comb.obs["method"].astype("category")
# Keep counts layer across the combined object:
if "counts" not in adata_comb.layers:
    adata_comb.layers["counts"] = adata_comb.X.copy()

In [ ]:
# -------------------- HVG (Seurat v3) on raw counts, then normalize/log --------------------
# HVG from raw counts avoids the non-integer warning.
sc.pp.highly_variable_genes(
    adata_comb,
    flavor="seurat_v3",
    n_top_genes=1000,
    batch_key="method",
    layer="counts"
)

# Normalize & log for downstream numeric analyses
sc.pp.normalize_total(adata_comb, target_sum=1e4)
sc.pp.log1p(adata_comb)

# Subset to the HVG feature set
adata_hv = adata_comb[:, adata_comb.var["highly_variable"]].copy()
features = adata_hv.var_names.to_list()  # same role as Seurat's SelectIntegrationFeatures

In [ ]:
# -------------------- Extract matrices by dataset, align by 'features' --------------------
adata_hv1 = adata_hv[adata_hv.obs["method"] == "data1"].copy()
adata_hv2 = adata_hv[adata_hv.obs["method"] == "data2"].copy()

# Ensure identical gene order (features) for X1/X2
adata_hv1 = adata_hv1[:, features].copy()
adata_hv2 = adata_hv2[:, features].copy()

X1 = adata_hv1.X
X2 = adata_hv2.X
X1 = X1.toarray() if hasattr(X1, "toarray") else np.asarray(X1)
X2 = X2.toarray() if hasattr(X2, "toarray") else np.asarray(X2)

# Keep cell type vectors, matching R variables ct.X1 / ct.X2
ct_X1 = adata_hv1.obs["cell_type"].astype(str).to_numpy()
ct_X2 = adata_hv2.obs["cell_type"].astype(str).to_numpy()

In [ ]:
# -------------------- Drop rare cell types (<20 per dataset), like your R keep.ind --------------------
def keep_mask(ct_array, min_count=20):
    counts = pd.Series(ct_array).value_counts()
    valid = set(counts[counts > min_count].index)
    return np.array([c in valid for c in ct_array], dtype=bool)

mask1 = keep_mask(ct_X1, min_count=20)
mask2 = keep_mask(ct_X2, min_count=20)

X1_f = X1[mask1]
X2_f = X2[mask2]
ct_X1_f = ct_X1[mask1]
ct_X2_f = ct_X2[mask2]

# Build a filtered combined obs table for saving
obs1 = adata_hv1.obs.iloc[mask1].copy()
obs2 = adata_hv2.obs.iloc[mask2].copy()
obs_out = pd.concat([obs1, obs2], axis=0)  # includes cell_type and method

In [ ]:
unique_cell_types, y  = np.unique(obs_out["cell_type"], return_inverse=True)
y1, y2 = y[obs_out["method"] == "data1"], y[obs_out["method"] == "data2"]

In [ ]:
# -------------------- Save using your res_path formalism --------------------
# Define res_path beforehand, e.g.:
# res_path = "/path/to/results"
# os.makedirs(res_path, exist_ok=True)

# Save X1/X2 after filtering (this mirrors where R ends up after filtering)
np.save(os.path.join(res_path, "X1.npy"), X1_f)                     # data1 (COV-5-D13)
np.save(os.path.join(res_path, "X2.npy"), X2_f)                     # data2 (COV-5-D1)
np.save(os.path.join(res_path, "dataset_names.npy"), np.array(["COV-5-D13", "COV-5-D1"]))

np.save(os.path.join(res_path, "labels1.npy"), y1)
np.save(os.path.join(res_path, "labels2.npy"), y2)
np.save(os.path.join(res_path, "cell_type_names.npy"), unique_cell_types)

pd.Series(features, name="gene").to_csv(os.path.join(res_path, "features.txt"), index=False)
obs_out.to_csv(os.path.join(res_path, "cells_metadata.csv"))        # filtered cells only
adata_hv.write(os.path.join(res_path, "combined_1000genes.h5ad"))   # combined AnnData (pre filter-by-frequency)

In [ ]:
X1.shape, X2.shape

In [ ]:
y1.max()

In [ ]:
len(np.unique(obs_out["cell_type"]))

In [ ]:
n0 = X1_f.shape[0]
n1 = X2_f.shape[0]



In [ ]:
cmap = plt.get_cmap("tab20")

color1 = np.array([cmap(i) for i in y1])
color2 = np.array([cmap(i) for i in y2])
colorj = np.concatenate((color1, color2))

In [ ]:
pca1_50 = PCA(n_components=50).fit_transform(X1_f)
pca2_50 = PCA(n_components=50).fit_transform(X2_f)
pcaj_50 = PCA(n_components=50).fit_transform(np.concatenate((X1_f, X2_f)))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].scatter(*pca1_50[:, :2].T, s=1, c="r")
ax[0].set_title("PCA2 on D13")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].set_title("PCA2 on D1")
ax[1].scatter(*pca2_50[:, :2].T, s=1, c="b")
ax[1].set_aspect("equal")
ax[1].axis("off")

ax[2].set_title("Joint PCA2")

order = np.random.permutation(pcaj_50.shape[0])
colors = ["r"] * n0 + ["b"] * n1
ax[2].scatter(*pcaj_50[order, :2].T, s=1, c=np.array(colors)[order])
ax[2].set_aspect("equal")
ax[2].axis("off")

fig.suptitle(f"PCA2 results of {dataset} dataset")

fig.savefig(os.path.join(fig_path, f"{dataset}_pca2.png"), dpi=300)


In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,5))
ax[0].scatter(*pca1_50[:, :2].T, s=1, c=color1)
ax[0].set_title("PCA2 on D13 data")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].set_title("PCA2 on D5 data")
ax[1].scatter(*pca2_50[:, :2].T, s=1, c=color2)
ax[1].set_aspect("equal")
ax[1].axis("off")

ax[2].set_title("Joint PCA2")

order = np.random.permutation(pcaj_50.shape[0])
colors = ["r"] * X1.shape[0] + ["b"] * X2.shape[0]
ax[2].scatter(*pcaj_50[order, :2].T, s=1, c=colorj[order])
ax[2].set_aspect("equal")
ax[2].axis("off")

handles = [
    mlines.Line2D(
        [], [], linestyle="none", marker="o",
        markersize=5, markerfacecolor=cmap(i ),
        markeredgecolor="none", label=str(lbl)
    )
    for i, lbl in enumerate(unique_cell_types)
]

ax[2].legend(handles=handles, title="Cell types", loc=(1.0, 0.5), fontsize=8)


fig.suptitle(f"PCA2 results of {dataset} dataset")

fig.savefig(os.path.join(fig_path, f"{dataset}_by_type_pca2.png"), dpi=300)

In [ ]:
tsne1 = TSNE(verbose=True).fit(pca1_50)
tsne2 = TSNE(verbose=True).fit(pca2_50)
tsnej = TSNE(verbose=True).fit(pcaj_50)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,5))

ax[0].scatter(*tsne1.T, s=1, c="r")
ax[0].set_title("tSNE on D13 data PCA50")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].set_title("tSNE on D5 PCA50")
ax[1].scatter(*tsne2.T, s=1, c="b")
ax[1].set_aspect("equal")
ax[1].axis("off")

ax[2].set_title("tSNE on joint PCA50")
colors = ["r"] * n0 + ["b"] * n1
order = np.random.permutation(len(colors))
ax[2].scatter(*tsnej[order].T, s=1, c=np.array(colors)[order])
ax[2].set_aspect("equal")
ax[2].axis("off")
fig.suptitle(f"tSNEs on {dataset} dataset")

fig.savefig(os.path.join(fig_path, f"{dataset}_tsne.png"), dpi=300)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,5))

ax[0].scatter(*tsne1.T, s=1, c=color1)
ax[0].set_title("tSNE on D13 data PCA50")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].set_title("tSNE on D5 PCA50")
ax[1].scatter(*tsne2.T, s=1, c=color2)
ax[1].set_aspect("equal")
ax[1].axis("off")

ax[2].set_title("tSNE on joint PCA50")
colors = ["r"] * n0 + ["b"] * n1
order = np.random.permutation(len(colors))
ax[2].scatter(*tsnej[order].T, s=1, c=colorj[order])
ax[2].set_aspect("equal")
ax[2].axis("off")
fig.suptitle(f"tSNEs on {dataset} dataset")

handles = [
    mlines.Line2D(
        [], [], linestyle="none", marker="o",
        markersize=5, markerfacecolor=cmap(i ),
        markeredgecolor="none", label=str(lbl)
    )
    for i, lbl in enumerate(unique_cell_types)
]

ax[2].legend(handles=handles, title="Cell types", loc=(1.0, 0.5), fontsize=8)

fig.savefig(os.path.join(fig_path, f"{dataset}_by_type_tsne.png"), dpi=300)

In [ ]:
y1, y2 = np.load(os.path.join(res_path, "labels1.npy")), np.load(os.path.join(res_path, "labels2.npy"))

In [ ]:
y1.max()

In [ ]:
cmap = plt.get_cmap("tab20")

color1 = np.array([cmap(i) for i in y1])
color2 = np.array([cmap(i) for i in y2])
colorj = np.concatenate((color1, color2))

In [ ]:
# diagonal variants

In [ ]:
diag_tsne = np.load("data/nCoV/nCoV_seed_0_epstimes_1_diag_True_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")["embd"]
P1 = np.load("data/nCoV/nCoV_seed_0_epstimes_1_diag_True_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")["P_sparse"]

In [ ]:
n0, n1 = P1.shape

In [ ]:
P1.min(), P1.max()

In [ ]:
plt.imshow(P1.T, interpolation="none", vmin=0)

In [ ]:
g = get_bipartite_graph(P1, diag=True).toarray()
#np.fill_diagonal(g, g.max())
g_plot = g[:n0,:n0].copy()
np.fill_diagonal(g_plot, g_plot.max())
plt.imshow(g_plot, interpolation="none")

In [ ]:
diag_tsne.shape

In [ ]:
colorj.shape

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

colors = ["r"] * n0 + ["b"] * n1
order = np.random.permutation(len(colorj))

ax[0].scatter(*diag_tsne[order].T, s=1, c=colorj[order])
ax[0].set_title("cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*diag_tsne[order].T, s=1, c=np.array(colors)[order])
ax[1].set_title("batch")
ax[1].set_aspect("equal")
ax[1].axis("off")

fig.suptitle(f"{dataset} OT-diagonal tSNE")

fig.savefig(os.path.join(fig_path, f"{dataset}_ot_diag_tsne_eps_1.png"), dpi=300)

In [ ]:
g_single = g[:n0, :n0].copy()

g_single /= g_single.sum()



In [ ]:
from openTSNE.affinity import Affinities
import scipy.sparse as sp

In [ ]:
embd_dim = 2
seed = 0
init = np.random.randn(g_single.shape[0], embd_dim) / 10000

In [ ]:
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=seed,
                early_exaggeration_iter=66,
                n_iter=133)
affinities = Affinities()
affinities.P = sp.csr_matrix(g_single)

embd_single = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(5,5))

order = np.random.permutation(n0)
print(order.dtype)

ax.scatter(*embd_single[order].T, s=1, c=colorj[order])
ax.set_title(f"{dataset} t-SNE on single diagonal block")
ax.set_aspect("equal")
ax.axis("off")
fig.savefig(os.path.join(fig_path, f"{dataset}_ot_diag_singleblock_tsne.png"), dpi=300)

In [ ]:
no_diag_tsne = np.load("data/nCoV/nCoV_seed_0_epstimes_1_diag_False_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")["embd"]

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

colors = ["r"] * X1_f.shape[0] + ["b"] * X2_f.shape[0]
order = np.random.permutation(len(colorj))

ax[0].scatter(*no_diag_tsne[order].T, s=1, c=colorj[order])
ax[0].set_title("OT-diagonal tSNE on joint PCA50, cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*no_diag_tsne[order].T, s=1, c=np.array(colors)[order])
ax[1].set_title("OT-diagonal tSNE on joint PCA50, batch")
ax[1].set_aspect("equal")
ax[1].axis("off")

fig.savefig(os.path.join(fig_path, f"{dataset}_ot_nodiag_tsne.png"), dpi=300)

In [ ]:
diag_100_tsne = np.load("data/nCoV/nCoV_seed_0_epstimes_100_diag_True_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")["embd"]
P_100 = np.load("data/nCoV/nCoV_seed_0_epstimes_100_diag_True_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")["P_sparse"]

In [ ]:
P_100.min(), P_100.max()

In [ ]:
plt.imshow(P_100.T, interpolation="none" )

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

colors = ["r"] * n0 + ["b"] * n1
order = np.random.permutation(len(colorj))

ax[0].scatter(*diag_100_tsne[order].T, s=1, c=colorj[order])
ax[0].set_title("OT-diagonal tSNE on joint PCA50, cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*diag_100_tsne[order].T, s=1, c=np.array(colors)[order])
ax[1].set_title("OT-diagonal tSNE on joint PCA50, batch")
ax[1].set_aspect("equal")
ax[1].axis("off")
fig.savefig(os.path.join(fig_path, f"{dataset}_ot_diag_tsne_eps_100.png"), dpi=300)


In [ ]:
## eps=10

In [ ]:
diag_10_tsne = np.load(
    "data/nCoV/nCoV_seed_0_epstimes_10_diag_True_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")[
    "embd"]
P_10 = np.load("data/nCoV/nCoV_seed_0_epstimes_10_diag_True_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")[
    "P_sparse"]

In [ ]:
P_10.min(), P_10.max()

In [ ]:
plt.imshow(P_10.T, interpolation="none")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

colors = ["r"] * n0 + ["b"] * n1
order = np.random.permutation(len(colorj))

ax[0].scatter(*diag_10_tsne[order].T, s=1, c=colorj[order])
ax[0].set_title("OT-diagonal tSNE on joint PCA50, cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*diag_10_tsne[order].T, s=1, c=np.array(colors)[order])
ax[1].set_title("OT-diagonal tSNE on joint PCA50, batch")
ax[1].set_aspect("equal")
ax[1].axis("off")

fig.savefig(os.path.join(fig_path, f"{dataset}_ot_diag_tsne_eps_10.png"), dpi=300)


In [ ]:
# sparsified eps=1

In [ ]:
diag_1_sparse_tsne = np.load(
    "data/nCoV/nCoV_seed_0_epstimes_1_diag_True_sparsify_None_norm_whole_sparsify_aff_True_dim_2_init_random_scale_10000_epochs_200_tsne.npz")[
    "embd"]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

colors = ["r"] * n0 + ["b"] * n1
order = np.random.permutation(len(colorj))

ax[0].scatter(*diag_1_sparse_tsne[order].T, s=1, c=colorj[order])
ax[0].set_title("cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*diag_1_sparse_tsne[order].T, s=1, c=np.array(colors)[order])
ax[1].set_title("batch")
ax[1].set_aspect("equal")
ax[1].axis("off")

fig.suptitle(f"{dataset} OT-diagonal sparse tSNE eps=1")

fig.savefig(os.path.join(fig_path, f"{dataset}_ot_diag_sparse_tsne_eps_1.png"), dpi=300)

In [ ]:
no_diag_1_sparse_tsne = np.load(
    "data/nCoV/nCoV_seed_0_epstimes_1_diag_False_sparsify_None_norm_whole_sparsify_aff_True_dim_2_init_random_scale_10000_epochs_200_tsne.npz")[
    "embd"]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

colors = ["r"] * n0 + ["b"] * n1
order = np.random.permutation(len(colorj))

ax[0].scatter(*no_diag_1_sparse_tsne[order].T, s=1, c=colorj[order])
ax[0].set_title("cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*no_diag_1_sparse_tsne[order].T, s=1, c=np.array(colors)[order])
ax[1].set_title("batch")
ax[1].set_aspect("equal")
ax[1].axis("off")

fig.suptitle(f"{dataset} non-diagonal sparse tSNE eps=1")

fig.savefig(os.path.join(fig_path, f"{dataset}_non_diag_sparse_tsne_eps_1.png"), dpi=300)

In [ ]:
# check plans

In [ ]:
sorted_idx0 = np.argsort(y1)
sorted_idx1 = np.argsort(y2)
sorted_idx = np.concatenate((sorted_idx0, sorted_idx1 + n0))

In [ ]:
plt.imshow(P1[np.ix_(sorted_idx0, sorted_idx1)], interpolation="none")
plt.colorbar()

In [ ]:
P1.shape

In [ ]:
len(y1)

In [ ]:
len(y2)

In [ ]:
P1.min()

In [ ]:
P1.max()

In [ ]:
g = get_bipartite_graph(P1, diag=True).toarray()

In [ ]:
plt.imshow(g[np.ix_(sorted_idx, sorted_idx)], interpolation="none")

In [ ]:
from ne_aligned_utils.utils import sparsify_affinities

In [ ]:
g_sparse = sparsify_affinities(g,n0=n0, n1=n1, k=15)

In [ ]:
plt.imshow(g_sparse[np.ix_(sorted_idx0, sorted_idx0)], interpolation="none")
plt.axis("off")
plt.savefig(os.path.join(fig_path, f"{dataset}_sparsified_ot_affinity_upper_diag.png"), dpi=300)

In [ ]:
embedder = TSNE(verbose=True, n_iter=1).fit(pca1_50)

In [ ]:
tsne_affinities = embedder.affinities.P.toarray()
tsne_bin_aff = (tsne_affinities > 0).astype(float)

In [ ]:
plt.imshow(tsne_affinities[np.ix_(sorted_idx0, sorted_idx0)], interpolation="none")
plt.colorbar()

In [ ]:
plt.imshow(np.log1p(10**7*tsne_affinities[np.ix_(sorted_idx0, sorted_idx0)]), interpolation="none")
plt.colorbar()


In [ ]:
plt.imshow(tsne_bin_aff[np.ix_(sorted_idx0, sorted_idx0)], interpolation="none")


In [ ]:
tsne_bin_aff.sum(1).min()

In [ ]:
from openTSNE.affinity import Uniform

In [ ]:
uni_aff = Uniform(pca1_50, k_neighbors=15, symmetrize="max").P.toarray()

In [ ]:
plt.imshow(uni_aff[np.ix_(sorted_idx0, sorted_idx0)], interpolation="none")
plt.axis("off")
plt.savefig(os.path.join(fig_path, f"{dataset}_uniform_kNN_affinity_upper_diag.png"), dpi=300)